# ASH-KV Real-World Observation Test
This notebook launches a live SGLang server in the background, pushes concurrent requests to force a cache eviction, and then queries the evicted prefixes to trigger a cache hit. 

By choking the KV cache capacity (`--mem-fraction-static 0.3`) and forcing `--hicache-write-policy write_through`, we force ASH-KV to intercept the IO and execute the Triton INT8 kernels.

In [ ]:
import subprocess
import time
import requests
import concurrent.futures
import os

# 1. Start the server in the background and redirect logs to a file
print("Starting SGLang server in the background...")
log_file = open("sglang_server.log", "w")

server_script = """
import ashkv.adapters.sglang.patcher
from sglang.launch_server import run_server
from sglang.srt.server_args import prepare_server_args
import sys
sglang_args = [
    '--model-path', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--port', '30000',
    '--mem-fraction-static', '0.3',
    '--hicache-write-policy', 'write_through',
    '--log-level', 'info'
]
server_args = prepare_server_args(sglang_args)
run_server(server_args)
"""

server_process = subprocess.Popen(
    ["python", "-c", server_script],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for the model to load
print("Waiting 20 seconds for the model to load and the server to boot...")
time.sleep(20)
print("Server should be ready! Logs are being written to 'sglang_server.log'")

In [ ]:
ENDPOINT = "http://localhost:30000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def make_prompt(i):
    return {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": f"Repeat this exactly {i}: " + "The quick brown fox jumps over the lazy dog. " * 200}
        ],
        "max_tokens": 50,
        "temperature": 0.0,
    }

# Phase 1: Fill the cache
print("=== Phase 1: Sending 5 requests to fill cache ===")
for i in range(5):
    try:
        r = requests.post(ENDPOINT, json=make_prompt(i))
        print(f"  Request {i}: {r.status_code} - {r.json()['choices'][0]['message']['content'][:60]}...")
    except Exception as e:
        print(f"  Request {i} FAILED: {e}")

In [ ]:
# Phase 2: Force eviction with concurrency
print("=== Phase 2: Sending 10 concurrent requests to force eviction ===")
print("SGLang's tiny cache (0.3 fraction) should overflow, triggering ASH-KV INT8 compression.")

def send_req(i):
    try:
        r = requests.post(ENDPOINT, json=make_prompt(i + 100), timeout=30)
        return f"  Request {i+100}: {r.status_code} - {r.json()['choices'][0]['message']['content'][:60]}..."
    except Exception as e:
        return f"  Request {i+100}: FAILED - {e}"

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:
    results = list(pool.map(send_req, range(10)))

for r in results:
    print(r)

# Give the background write_through threads a second to finish compressing
time.sleep(3)

In [ ]:
# Phase 3: Force Cache Hits (Decompression)
print("=== Phase 3: Sending requests that should hit the compressed cache ===")
print("These requests share prefixes with Phase 1. SGLang will pull them from CPU (HiCache), triggering ASH-KV INT8 decompression.")

for i in range(5):
    try:
        r = requests.post(ENDPOINT, json=make_prompt(i))
        print(f"  Repeat {i}: {r.status_code} - {r.json()['choices'][0]['message']['content'][:60]}...")
    except Exception as e:
        print(f"  Repeat {i} FAILED: {e}")

In [ ]:
print("=== Analyzing ASH-KV Logs ===")
log_file.flush()

with open("sglang_server.log", "r") as f:
    logs = f.readlines()
    
ashkv_logs = [line.strip() for line in logs if "[ASH-KV]" in line]

if ashkv_logs:
    for line in ashkv_logs:
        print(line)
else:
    print("No ASH-KV logs found. The patches might not have been applied or eviction wasn't triggered.")

print("\n=== Full Server Logs (Last 20 Lines) ===")
for line in logs[-20:]:
    print(line.strip())

In [ ]:
# Cleanup the server process
server_process.terminate()
server_process.wait()
log_file.close()
print("Server shut down successfully.")